# Paper 1: current sheets at beta_p = beta_e = 1, with the k-cutoff filter

1. The B spectrum before and after the filter, with the wavenumbers that matter marked.
2. The thickness of every sheet (M1, M2, M3), pooled over the snapshots, with lognormal fits.
3. The length and the aspect ratio of every sheet, pooled the same way.

`pta.sheets.pooled_thickness` and `pooled_shape` do the filtering, detection, measurement, pooling and fitting,
and cache the filtered fields; this notebook says which run, which snapshots and which filter. It replaces the
two 300-line copies of that pipeline the notebook used to hold.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from pta import config
from pta.plotting import save_figure, use_style
from pta.sheets import filtered_field, pooled_shape, pooled_thickness
from pta.spectra import summed_spectrum

use_style()
RESULTS = config.results_dir("paper1_methods", "CSDetectionKCutoffBeta1")

In [ ]:
BETA_P = BETA_E = 1
BRMS = 0.24
TIMES = range(200, 250, 5)            # the snapshots pooled for the thickness
SHAPE_TIMES = range(40, 50)           # the snapshots pooled for the length and aspect ratio

K_CUTOFF = 14.12 / 3                  # a third of the k d_i where the density SNR drops to 3
XI = 1.6

# wavenumbers marked on the spectrum (from quality_checks/SNR and the pooled medians below)
SNR3_K = 14.1252
SPECTRUM_TIME = 200
K_RANGE_SHOWN = (5e-2, 30)
POWER_RANGE_SHOWN = (1e-6, 1)

METHODS = ("m1", "m2", "m3")
N_BOOTSTRAP = 1000

In [ ]:
# ---- the B spectrum, before and after the filter ----
run = config.run("beta_scan", beta_p=BETA_P, beta_e=BETA_E, brms=BRMS)
B = run.load_many(["Bx", "By", "Bz"], SPECTRUM_TIME)
k, spectrum = summed_spectrum(B, run.dx, run.dy)
B_filtered = filtered_field(run, SPECTRUM_TIME, ("Bx", "By", "Bz"), k=K_CUTOFF, Xi=XI)
_, spectrum_filtered = summed_spectrum(B_filtered, run.dx, run.dy)

fig, ax = plt.subplots()
ax.loglog(k, spectrum, label="B spectrum")
ax.loglog(k, spectrum_filtered, linestyle=":", label="B spectrum with filter")
ax.axvline(SNR3_K, linestyle="--", color="red", label=r"$k d_i$ at SNR = 3")
ax.axvline(K_CUTOFF, linestyle="--", color="blue", label="k cutoff")
ax.set_xlabel(r"$k$")
ax.set_ylabel(r"$P(k)$")
ax.set_xlim(*K_RANGE_SHOWN)
ax.set_ylim(*POWER_RANGE_SHOWN)
ax.legend()
save_figure(fig, RESULTS, f"B_spectrum_filtered_t{SPECTRUM_TIME}")
plt.show()

In [ ]:
# ---- thickness of every sheet, pooled over TIMES ----
pooled = pooled_thickness(beta_p=BETA_P, beta_e=BETA_E, brms=BRMS, times=TIMES, k=K_CUTOFF, Xi=XI,
                          method=METHODS, bootstrap=N_BOOTSTRAP)
figure_path, array_path = pooled.save(RESULTS, "pooled_thickness")
for row in pooled.statistics:
    print(f"{row['method']}: {row['n']} sheets, lognormal median {row['median_lognorm']:.4f} d_p "
          f"[{row['ci_lo']:.4f}, {row['ci_hi']:.4f}], sigma {row['sigma_lognorm']:.3f}")
plt.show()

In [ ]:
# ---- length and aspect ratio of every sheet, pooled over SHAPE_TIMES ----
shape = pooled_shape(beta_p=BETA_P, beta_e=BETA_E, brms=BRMS, times=SHAPE_TIMES, k=K_CUTOFF, Xi=XI)
figure_path, array_path = shape.save(RESULTS, "shape")
lost = shape.measurable - len(shape.length)
print(f"{len(shape.length)} sheets pooled from {len(shape.times)} snapshots; "
      f"{lost} of {shape.measurable} ({100 * lost / max(shape.measurable, 1):.1f}%) had no usable M3 fit")
print(f"length      : median {np.median(shape.length):6.2f} d_p")
print(f"thickness   : median {np.median(shape.thickness):6.3f} d_p  (M3)")
print(f"aspect ratio: median {np.median(shape.aspect):6.1f}")
plt.show()